## Week 8 Day 5 — AI News Digest (Multi-Agent Pipeline)

This is a **multi-agent news digest system** with a live-streaming Gradio UI.

### What it does
1. The user selects one or more news categories (science, technology, showbiz, war, etc.) from a checkbox panel.
2. Clicking **Run Pipeline** kicks off a four-agent pipeline that runs in a background thread and streams results to the UI in real time:
   - **Category Agent** — queries Google News RSS to discover the latest article titles and publisher domains for each selected category.
   - **Extractor Agent** — resolves the real article URL via a DuckDuckGo site-search, then downloads and parses the full article text using `newspaper3k` (with a `BeautifulSoup` fallback).
   - **Summariser Agent** — calls GPT-4o-mini to produce a 3-sentence summary of each article; falls back to a sentence-extraction heuristic if no API key is available.
   - **Coordinator** — orchestrates the agents across all selected categories and streams `status` and `article` events back to the UI.
3. The UI updates live as each article is processed — showing the active agent, a status bar log, and the growing digest of summaries with links.

In [ ]:
import gradio as gr
from agents.coordinator import NewsCoordinator

AVAILABLE_CATEGORIES = ["science", "technology", "showbiz", "war", "business", "health", "sports", "environment"]

CATEGORY_ICONS = {
    "science": "🔬",
    "technology": "💻",
    "showbiz": "🎬",
    "war": "⚔️",
    "business": "💼",
    "health": "🏥",
    "sports": "⚽",
    "environment": "🌿",
}

AGENT_CSS = open('styles.css','r').read()

AGENT_NAMES = [
    ("🔍", "Category Agent",   "Discovers articles from Google News RSS"),
    ("📥", "Extractor Agent",  "Downloads & parses article content"),
    ("✍️", "Summariser Agent", "Generates AI-powered summaries"),
    ("🎯", "Coordinator",      "Orchestrates the full pipeline"),
]

def _pill(icon: str, name: str, state: str) -> str:
    return (
        f'<div class="agent-pill {state}">'
        f'<span class="dot"></span>{icon} {name}'
        f'</div>'
    )

def _pipeline_html(active_index: int | None = None, done_indices: set | None = None) -> str:
    done_indices = done_indices or set()
    pills = []
    for i, (icon, name, _) in enumerate(AGENT_NAMES):
        if i in done_indices:
            state = "done"
        elif i == active_index:
            state = "active"
        else:
            state = "idle"
        pills.append(_pill(icon, name, state))
    return '<div class="agent-pipeline">' + "".join(pills) + "</div>"


def run_pipeline(selected_categories):
    if not selected_categories:
        yield (
            _pipeline_html(),
            '<div class="status-bar">⚠️ Please select at least one category.</div>',
            "",
        )
        return

    done = set()
    pipeline_html = _pipeline_html(active_index=3)       # Coordinator active
    status = '<div class="status-bar">🚀 Starting News Coordinator…</div>'
    output = ""
    yield pipeline_html, status, output

    coordinator = NewsCoordinator(selected_categories)
    current_cat = None
    article_count = 0

    for event in coordinator.stream():
        if event["type"] == "status":
            cat = event["category"]
            msg = event["message"]
            icon = CATEGORY_ICONS.get(cat, "📰")

            if "Searching" in msg:
                pipeline_html = _pipeline_html(active_index=0, done_indices=done)
                status = f'<div class="status-bar">{icon} <b>{cat.title()}</b> — {msg}</div>'
            elif "Found" in msg:
                pipeline_html = _pipeline_html(active_index=1, done_indices=done)
                status = f'<div class="status-bar">{icon} <b>{cat.title()}</b> — {msg}</div>'
            else:
                status = f'<div class="status-bar">ℹ️ {msg}</div>'

            yield pipeline_html, status, output

        elif event["type"] == "article":
            cat    = event["category"]
            title  = event["title"]
            url    = event["url"]
            summary = event.get("summary") or "No summary available."
            icon   = CATEGORY_ICONS.get(cat, "📰")

            # Summariser active while we just received the article
            pipeline_html = _pipeline_html(active_index=2, done_indices=done)
            status = f'<div class="status-bar">✍️ Summarised: <i>{title[:80]}…</i></div>'

            if cat != current_cat:
                current_cat = cat
                output += f"\n## {icon} {cat.upper()}\n\n"

            article_count += 1
            output += f"**{article_count}. {title}**\n\n{summary}\n\n🔗 [{url}]({url})\n\n---\n\n"
            yield pipeline_html, status, output

    done = {0, 1, 2, 3}
    pipeline_html = _pipeline_html(done_indices=done)
    status = f'<div class="status-bar">✅ Done! Fetched {article_count} article(s) across {len(selected_categories)} category/categories.</div>'
    yield pipeline_html, status, output


# UI layout
def build_ui():
    with gr.Blocks(css=AGENT_CSS, title="AI News Digest", theme=gr.themes.Base()) as demo:

        gr.HTML("""
        <div id="header-banner">
          <h1>📡 AI News Digest</h1>
          <p>Multi-agent pipeline powered by GPT-4o-mini — select categories and watch the agents work live.</p>
        </div>
        """)

        with gr.Row():
            with gr.Column(scale=1, min_width=280):
                gr.Markdown("### 🗂️ Select Categories")
                category_checks = gr.CheckboxGroup(
                    choices=AVAILABLE_CATEGORIES,
                    value=["science", "technology"],
                    label="",
                    elem_id="category-group",
                )
                gr.Markdown("### 🤖 Agent Pipeline")
                pipeline_display = gr.HTML(_pipeline_html())
                status_display   = gr.HTML('<div class="status-bar">Waiting to start…</div>')

                with gr.Row():
                    run_btn   = gr.Button("▶  Run Pipeline", elem_id="run-btn", variant="primary")
                    clear_btn = gr.Button("✕ Clear", elem_id="clear-btn")

            with gr.Column(scale=3):
                gr.Markdown("### 📰 Live Results")
                results = gr.Markdown(
                    value="*Results will stream here as agents complete their work…*",
                    elem_id="results-panel",
                )

        # agent descriptions accordion
        with gr.Accordion("ℹ️ How the agent pipeline works", open=False):
            agent_rows = "".join(
                f"| {icon} **{name}** | {desc} |\n"
                for icon, name, desc in AGENT_NAMES
            )
            gr.Markdown(
                "| Agent | Role |\n|---|---|\n" + agent_rows
            )

        # event wiring
        run_btn.click(
            fn=run_pipeline,
            inputs=[category_checks],
            outputs=[pipeline_display, status_display, results],
            stream_every=0.1,
        )
        clear_btn.click(
            fn=lambda: (_pipeline_html(), '<div class="status-bar">Cleared.</div>', ""),
            inputs=[],
            outputs=[pipeline_display, status_display, results],
        )

    return demo


app = build_ui()
app.launch(inbrowser=True)